# 06 — AI Client
Test `generate_text` and `generate_json` via `AnthropicClient`.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from src.config import get_anthropic_settings
from src.clients.anthropic_client import AnthropicClient

settings = get_anthropic_settings()
print(settings.masked())

client = AnthropicClient(settings)

{'api_key': 'sk-ant-a…***', 'model_haiku': 'claude-haiku-4-5-20251001', 'model_sonnet': 'claude-sonnet-4-6'}


## generate_text — Haiku (default)

In [3]:
text = client.generate_text(
    system_prompt="You are a concise risk analyst.",
    user_prompt="In one sentence, what does it mean when many companies share the same registered address?",
)
print(text)

It typically indicates a shared office space, mail forwarding service, or potentially shell companies—raising concerns about legitimacy if combined with other red flags.


## generate_text — Sonnet override

In [4]:
text_sonnet = client.generate_text(
    system_prompt="You are a concise risk analyst.",
    user_prompt="In two sentences, explain what Ultimate Beneficial Ownership (UBO) means in a corporate context.",
    model=settings.model_sonnet,
)
print(text_sonnet)

Ultimate Beneficial Ownership (UBO) refers to the natural person(s) who ultimately own or control a company, either through direct shareholding or through a chain of ownership or control, regardless of how many intermediate entities exist. Identifying UBOs is critical in compliance and anti-money laundering (AML) frameworks to prevent corporate structures from being used to conceal the true individuals who benefit from or control a business.


## generate_json — structured output

In [5]:
import json

result = client.generate_json(
    system_prompt="You are a risk assessment engine.",
    user_prompt=(
        "A company called 'ACME HOLDINGS LTD' has 47 other companies registered "
        "at the same address and its only owner is an offshore company. "
        "Return a JSON object with keys: risk_level (low/medium/high), "
        "risk_factors (list of strings), and summary (one sentence)."
    ),
)

print(json.dumps(result, indent=2))

{
  "risk_level": "high",
  "risk_factors": [
    "Multiple companies registered at same address (47 others) - indicates possible shell company network",
    "Sole ownership by offshore company - lack of transparency and beneficial ownership obscurity",
    "Centralized address for numerous entities - suggests potential coordinated or fictitious operations",
    "Offshore ownership structure - difficulty in regulatory oversight and beneficial owner identification",
    "Concentration of companies at single location - elevated money laundering and fraud risk"
  ],
  "summary": "ACME HOLDINGS LTD presents high risk due to offshore ownership structure combined with 47 co-registered companies at the same address, suggesting a potential shell company network with obscured beneficial ownership."
}


## generate_json — validate schema

In [6]:
assert "risk_level" in result, "missing risk_level"
assert "risk_factors" in result, "missing risk_factors"
assert isinstance(result["risk_factors"], list), "risk_factors should be a list"
print(f"risk_level  : {result['risk_level']}")
print(f"risk_factors: {result['risk_factors']}")
print(f"summary     : {result['summary']}")

risk_level  : high
risk_factors: ['Multiple companies registered at same address (47 others) - indicates possible shell company network', 'Sole ownership by offshore company - lack of transparency and beneficial ownership obscurity', 'Centralized address for numerous entities - suggests potential coordinated or fictitious operations', 'Offshore ownership structure - difficulty in regulatory oversight and beneficial owner identification', 'Concentration of companies at single location - elevated money laundering and fraud risk']
summary     : ACME HOLDINGS LTD presents high risk due to offshore ownership structure combined with 47 co-registered companies at the same address, suggesting a potential shell company network with obscured beneficial ownership.
